# HPC4e heterogeneous mixed elasticity

Devloo et al. (2021), section 6.2, DOI [10.1051/m2an/2021013](https://doi.org/10.1051/m2an/2021013).
The original geological material samples, 16 × 8 macrorectangles, RT1/Q1/P1 local spaces and 32 × 32 local rectangles match the published experiment. Only the number of P1 subfaces changes: 1, 2, 4, 8. The expected stress-profile oscillations decrease under this skeleton refinement.

This notebook reads the archived research acquisitions; the final cell solves a small affine mixed-elasticity patch. Full geological solves and classical RT reference refinement are separate from lightweight automated tests.

The main equations execute from source and state their current discretization. The original archived comparisons below execute with `--historical` and require their exact attributed payloads. `--study` runs the complete public current acquisition declared in the catalogue, with its own output provenance.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/b2dfafe24e1cd43ef28f3b2df0676b01a3352fe44819ef8be01fd9a0854deba4/50_hpc4e_geomechanics-companion.zip"
COMPANION_SHA256 = "b2dfafe24e1cd43ef28f3b2df0676b01a3352fe44819ef8be01fd9a0854deba4"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("elasticity/50_hpc4e_geomechanics.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## Current full material section through public strain equations

This acquisition downloads the pinned original material files with verified
checksums. It uses every one of the 512×256 material pixels on the complete
10000 m × 4500 m section. Material values and gravity retain the original
scaling. The displacement is zero at the bottom and physical outward traction
is zero on the remaining exterior faces.

The independent reference uses continuous displacement Q1 and finite
plane-strain stiffness. Its approximation cells may cross material interfaces;
quadrature cuts each cell at the exact pixel intersections. This preserves the
coefficient and source fields without replacing them by cell averages. This
reference space differs from the RT1/Q1/P1 weak-stress space used in the paper.
The three quick levels measure their own displacement and raw-gradient L2
increments. They are insufficient as a converged baseline when those increments
remain large. The fitted weak-stress acquisition and finer classical mixed
references are reproduced by the full `--study` recipe.

$$
\begin{aligned}
a(u,v)&=\int_\Omega \varepsilon(v):C(x):\varepsilon(u),\\
L(v)&=\int_\Omega f(x)\cdot v,
&f&=\frac{10000}{10^8}(0,-9.81\rho),\\
u\vert_{\Gamma_{\mathrm{bottom}}}&=0,
&(C\varepsilon(u))n\vert_{\Gamma_{\mathrm{free}}}&=0.
\end{aligned}
$$


In [ ]:
import numpy as np
from threadpoolctl import threadpool_limits
from examples.hpc4e_data import BOUNDS, load_data
from examples.formulations.conforming_elasticity import (
    conforming_elasticity, displacement_difference,
)
from pymhm.meshes.cartesian import CartesianMacroMesh

material = load_data(download=True)
lame_lambda, lame_mu, gravity = material.fields()
current_references = []
with threadpool_limits(1):
    for nx, ny in ((16, 8), (32, 16), (64, 32)):
        reference_mesh = CartesianMacroMesh(nx, ny, BOUNDS)
        midpoints = reference_mesh.points[reference_mesh.faces].mean(axis=1)
        free_faces = tuple(
            int(face) for face in reference_mesh.boundary_faces
            if not np.isclose(midpoints[face, 1], 0.0, rtol=0, atol=1e-14)
        )
        field, residual = conforming_elasticity(
            reference_mesh, degree=1, lame_lambda=lame_lambda, lame_mu=lame_mu,
            source=gravity, integration_field=lame_lambda, free_faces=free_faces,
            dirichlet=(0.0, 0.0), order=2,
        )
        assert residual < 1e-10
        current_references.append(field)
    increments = [
        displacement_difference(fine, coarse)
        for coarse, fine in zip(current_references[:-1], current_references[1:], strict=True)
    ]
print("Material samples:", material.young.shape, "; full dimensionless bounds:", BOUNDS)
print("Independent Q1 increments (fine norm denominators):", increments)
print("Large reference increments require further refinement before baseline use.")


In [ ]:
if HISTORICAL_REPLAY:
    import json
    from pathlib import Path
    from IPython.display import Image, display

    records = root / "examples/results/hpc4e"
    manifest = json.loads((records / "dataset.json").read_text())
    print(manifest["repository"], manifest["revision"])
    print("Material grid:", manifest["shape"], "; bounds [m]:", manifest["physical_bounds_m"])
    for segments in (1, 2, 4, 8):
        row = json.loads((records / f"mhm-s{segments}.json").read_text())
        print(segments, "subfaces:", row["unknown_global_dofs"], "global unknowns", row["diagnostics"])

## Published profile and independent reference

The profile is at z = 2250.25 m. Digitization intervals contain the visible line thickness plus two pixels vertically and horizontally. Their overlap checks agreement with the image, not an integrated error bound. Independently assembled DOLFINx/UFL RT1 and RT2 fields on 512 × 256 cells check polynomial-degree sensitivity; RT2 on 1024 × 512 checks spatial refinement. The fine RT2 acquisition uses four MPI ranks and pivoted MUMPS LDLᵀ with out-of-core factors. Weak stress symmetry is a moment constraint; all stress norms use the full tensor.


In [ ]:
if HISTORICAL_REPLAY:
    import math
    comparison = json.loads((records / "complementary-energy.json").read_text())
    for row in comparison["rows"]:
        energy = row["energy"]
        print("subfaces", row["segments"], "stress L2 [%]", 100 * row["l2"]["stress_relative"],
              "displacement L2 [%]", 100 * row["l2"]["displacement_relative"],
              "compliance [%]", 100 * math.sqrt(
                  energy["stress_compliance_difference_squared"] / energy["reference_energy"]),
              "energy identity relative defect", energy["identity_relative_difference"])
    comparisons = json.loads((records / "published-profile-comparison.json").read_text())
    for row in comparisons:
        print("subfaces", row["segments"], "overlap", row["interval_overlap_count"], "/", row["samples"],
              "nominal RMS [MPa]", row["nominal_rms_difference_mpa"])
    for filename in ("material.png", "published-profile-comparison.png", "stress-profiles.png",
                     "stress-displacement-fields.png", "displacement-vectors.png"):
        display(Image(filename=str(root / "docs/figures/hpc4e" / filename)))

## Classical reference sensitivity

The RT2 spatial increment from 512 × 256 to 1024 × 512 is 1.119129% in stress L2, 0.010589% in displacement L2, 2.864135% in rotation L2 and 3.113395% in compliance norm. These increments use the fine reference norm as denominator and are not continuum error bounds. The finest MHM field differs from that reference by 4.159399%, 0.625204%, 8.202742% and 7.338907%, respectively. Reference sensitivity remains relevant, particularly for rotation and compliance.

The separate RT1-to-RT2 degree increment on 512 × 256 is retained in its own numerical record. Orders four and five check all field-norm integrals on the common, material-aligned partition. The figure manifest ties the displayed images to the reference and MHM input digests.


In [ ]:
if HISTORICAL_REPLAY:
    import hashlib

    refinement = json.loads((records / "reference-spatial-refinement.json").read_text())
    assert refinement["source_changed"] is False
    pairs = {}
    for row in refinement["rows"]:
        pairs.setdefault(row["approximation"], {})[row["norms"]["quadrature_order"]] = row["norms"]
    for name, pair in pairs.items():
        for field in ("stress", "displacement", "rotation", "compliance"):
            key = field + "_difference_dimensionless"
            assert abs(pair[4][key] - pair[5][key]) <= 1e-10 * pair[5][key]
        print(name, {field: 100 * pair[5][field + "_relative"]
              for field in ("stress", "displacement", "rotation", "compliance")})
    for name in ("classical-rt1-512x256-pardiso", "classical-rt2-512x256-mumps",
                 "classical-rt2-1024x512-mumps"):
        reference = json.loads((records / (name + ".json")).read_text())
        print(name, "DOFs", reference["native_dofs"], "original residual", reference["residual"],
              "force equilibrium", reference["equilibrium_relative"],
              "energy/work", reference["energy_work_relative"])
    visualization = json.loads((records / "visualization.json").read_text())
    assert visualization["reference"] == "classical-rt2-1024x512-mumps.npz"
    assert visualization["input_sha256"][visualization["reference"]] == reference["archive_sha256"]
    for name, expected in visualization["figure_sha256"].items():
        assert hashlib.sha256((root / "docs/figures/hpc4e" / name).read_bytes()).hexdigest() == expected
    print("Six figure digests verified; field samples preserve owning-cell values.")
    display(Image(filename=str(root / "docs/figures/hpc4e/reference-sensitivity.png")))


## Small exact patch

The affine displacement u = (x, −z) is represented exactly. Its trace-free strain makes the field compatible with both finite and incompressible bulk response; this patch checks the operator and the three rigid modes, not general locking-free convergence.

## User-defined weak-symmetry block equations

The primary solve declares the actual mixed coefficient fields
$(\sigma,u,q,\psi)$ through `LocalEquations`. Shared kernels integrate stress
compliance, divergence, asymmetry, physical force, rigid moments and normal
maps. The local matrix is

$$
\begin{bmatrix}
 A&\mathrm{Div}^\top&\mathrm{Asym}^\top&-S\\
 \mathrm{Div}&0&0&0\\
 \mathrm{Asym}&0&0&0\\
 -S^\top&0&0&0
\end{bmatrix}
\begin{bmatrix}\sigma\\u\\q\\\psi\end{bmatrix}
+\begin{bmatrix}0\\0\\0\\-T\end{bmatrix}\lambda
=\begin{bmatrix}0\\-f\\0\\0\end{bmatrix}.
$$

$S$ selects local boundary normal moments and $T$ maps the canonical macro
traction basis to those moments, including the incident-cell orientation.
The three exact rigid modes and their physical displacement moments are
explicit. With $B=(0,0,0,-T)^\top$, the declared global pairing is $C=-B^\top$,
so the global boundary load is the **positive** displacement moment in this
coordinate convention. `Equation` supplies that load and zero extra load in
the retained rigid coordinates. The full-displacement hydrostatic constraint
uses the independently integrated physical compliance trace identity.

The same local/global core eliminates these declared fields. Physical field
records retain stress/displacement/rotation evaluation, norms and equilibrium
checks. Solver names do not select the primary forms.

In [ ]:
import inspect
import numpy as np
from IPython.display import Code
from pymhm.meshes.cartesian import CartesianMacroMesh
from examples.tutorial_stress_equations import (
    solve_tensor, mixed_equations, tensor_equations, global_equation,
)

for builder in (mixed_equations, tensor_equations, global_equation):
    display(Code(inspect.getsource(builder), language="python"))

mesh = CartesianMacroMesh(2, 1, (0, 1, 0, 0.45))

def exact(points: np.ndarray) -> np.ndarray:
    """Original trace-free affine displacement in the physical rectangle."""
    return np.column_stack((points[:, 0], -points[:, 1]))

system, solution = solve_tensor(
    mesh, local_refinement=2, dirichlet=exact, lame_lambda=1e8,
)
assert np.max(np.abs(solution.equilibrium_residuals())) < 1e-9
assert np.max(np.abs(solution.fine_force_residuals())) < 1e-9
print("Force and rigid-moment equilibrium:", np.max(np.abs(solution.equilibrium_residuals())))

## Explicit compatibility comparison

The same RT1/Q1/P1 patch is solved separately by the earlier compatibility
solver. All mixed coefficient fields, trace and retained rigid amplitudes are
compared in their unchanged bases. The full stress, displacement, divergence
and rotation errors use the original physical fields. This inexpensive patch
checks those forms and modes; the geological acquisitions above remain
separate and retain their original material and reference provenance.

In [ ]:
from examples.formulations.application import weak_stress_elasticity as solve_elasticity_tensor_rt

reference = solve_elasticity_tensor_rt(
    mesh, local_refinement=2, dirichlet=exact, lame_lambda=1e8,
)
for own, previous in zip(
    (*solution.hybrid.fields, solution.hybrid.trace, *solution.hybrid.coarse),
    (*reference.hybrid.fields, reference.hybrid.trace, *reference.hybrid.coarse), strict=True,
):
    np.testing.assert_allclose(own, previous, atol=1e-12, rtol=1e-10)
errors = solution.errors(exact, np.diag([2.0, -2.0]), (0.0, 0.0), 0.0)
reference_errors = reference.errors(exact, np.diag([2.0, -2.0]), (0.0, 0.0), 0.0)
np.testing.assert_allclose(list(errors.values()), list(reference_errors.values()), atol=1e-12, rtol=1e-10)
print(errors)
print("Original RT stress/displacement/rotation and rigid coefficients agree within atol=1e-12 and rtol=1e-10")